# Install libraries

In [ ]:
pip install gplearn

# Import modules and libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
project_root = os.path.abspath('/kaggle/input/datasets/phmhuyphongp/modules-vn')
if project_root not in sys.path:
    sys.path.append(project_root)

import src.models as md
import src.evaluation as ev
import src.features as ft
from config import candidate_features, BASE_MODEL_PARAMS

# Preprocessing

In [ ]:
df = pd.read_parquet('/kaggle/input/datasets/phmhuyphongp/qualified-stocks-for-researching/market_data_filtered.parquet')

## Std (Almgren et al)

In [ ]:
import numpy as np
import pandas as pd

# Load the dataset
df = pd.read_parquet('/kaggle/input/datasets/phmhuyphongp/qualified-stocks-for-researching/market_data_filtered.parquet')

# 1. Sort data chronologically within each stock
#    (replace 'date' below if your time column has a different name)
df = df.sort_values(by=['Symbol', 'date'])

# 2. Compute daily log returns, separately for each stock
df['log_return'] = df.groupby('Symbol')['close'].transform(
    lambda x: np.log(x / x.shift(1))
)

# 3. Compute rolling 20-day volatility (std dev of log returns), separately for each stock
df['sigma_daily'] = df.groupby('Symbol')['log_return'].transform(
    lambda x: x.rolling(window=20).std(ddof=1)
)

# 4. Get each stock's average volatility over the full sample period
avg_sigma_per_stock = df.groupby('Symbol')['sigma_daily'].mean()

# Identify the stock with the highest average volatility
most_volatile_stock = avg_sigma_per_stock.idxmax()
highest_avg_sigma = avg_sigma_per_stock.max()

# Average volatility across all stocks in the dataset
mean_sigma_all_stocks = avg_sigma_per_stock.mean()

print("Stock with the highest average volatility:")
print(f"  {most_volatile_stock}: {highest_avg_sigma:.4%}")

print("\nAverage volatility across all stocks in the dataset:")
print(f"  {mean_sigma_all_stocks:.4%}")

In [ ]:
gamma = 0.314
eta = 0.142
beta = 0.6

# take the max average daily volatility from your data (as a decimal, not %)
sigma_max = avg_sigma_per_stock.max()  # e.g. 0.045 for 4.5%

# assumed trade size as a fraction of daily volume, e.g. 1%
X_over_VT = 0.01

# temporary component (ignoring permanent impact I, since paper notes I is small)
J_temp = eta * sigma_max * (X_over_VT ** beta)

print(f"Estimated slippage: {J_temp*100:.4f}%")

## Start and End of each stock

In [ ]:
df["date"] = pd.to_datetime(df["date"])
summary = (
    df.groupby("Symbol")["date"]
    .agg(**{"First Date": "min", "Last Date": "max"})
    .reset_index()
)


In [ ]:
summary.to_csv("/kaggle/working/symbol_dates.csv", index=False)

## Number of stock per date

In [ ]:
import matplotlib.pyplot as plt

# Your groupby result
symbol_counts = df.groupby('date')['Symbol'].nunique()

# Plot
fig = plt.figure(figsize=(12, 6))
symbol_counts.plot(kind='line')
plt.title('Number of Unique Symbols per Date')
plt.xlabel('Date')
plt.ylabel('Unique Symbol Count')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()


In [ ]:
fig.savefig('/kaggle/working/symbol_counts_per_date_before.png', dpi=300, bbox_inches='tight')

Removing 2 days missing majority of stocks

In [ ]:
df['date'] = pd.to_datetime(df['date'])

excluded_dates = pd.to_datetime(['2018-01-23', '2018-01-24'])

df = df[~df['date'].isin(excluded_dates)].copy()

In [ ]:
import matplotlib.pyplot as plt

# Your groupby result
symbol_counts = df.groupby('date')['Symbol'].nunique()

# Plot
fig = plt.figure(figsize=(12, 6))
symbol_counts.plot(kind='line')
plt.title('Number of Unique Symbols per Date')
plt.xlabel('Date')
plt.ylabel('Unique Symbol Count')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
fig.savefig('/kaggle/working/symbol_counts_per_date_after.png', dpi=300, bbox_inches='tight')

# Build features

In [ ]:
df = ft.build_features(df = df,min_stocks_per_date=0,adtv_limit = 2_500_000)
df.shape

## Demonstration for Walk Forward

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from matplotlib.patches import Patch

# ============================================================
# 1. Generate folds
# ============================================================

folds = md.generate_folds(
    df=df,
    initial_train_months=24,
    test_months=6,
    gap_days=21,
)

rows = []

for f in folds:
    rows.append({
        "fold":        f.fold_number,
        "train_start": f.train_df["date"].min(),
        "train_end":   f.train_df["date"].max(),
        "test_start":  f.test_start,
        "test_end":    f.test_df["date"].max(),
    })

fold_dates = pd.DataFrame(rows)

# Save fold information
fold_dates.to_csv(
    "/kaggle/working/fold_dates.csv",
    index=False
)


# ============================================================
# 2. Create figure
# ============================================================

fig, ax = plt.subplots(figsize=(14, 8))

bar_height = 0.55

for _, row in fold_dates.iterrows():

    y = row["fold"]

    # Convert dates to matplotlib's numerical date representation
    train_start = mdates.date2num(row["train_start"])
    train_end   = mdates.date2num(row["train_end"])
    test_start  = mdates.date2num(row["test_start"])
    test_end    = mdates.date2num(row["test_end"])

    # --------------------------------------------------------
    # Training period
    # --------------------------------------------------------

    ax.barh(
        y,
        train_end - train_start,
        left=train_start,
        height=bar_height,
        color="lightgray",
        edgecolor="black",
        linewidth=0.8,
        label="Giai đoạn huấn luyện" if y == 1 else None
    )

    # --------------------------------------------------------
    # Gap
    # --------------------------------------------------------

    ax.barh(
        y,
        test_start - train_end,
        left=train_end,
        height=bar_height,
        color="white",
        edgecolor="black",
        linewidth=0.8,
        linestyle="--",
        label="Khoảng trống" if y == 1 else None
    )

    # --------------------------------------------------------
    # Testing period
    # --------------------------------------------------------

    ax.barh(
        y,
        test_end - test_start,
        left=test_start,
        height=bar_height,
        color="white",
        edgecolor="black",
        linewidth=0.8,
        label="Giai đoạn kiểm định" if y == 1 else None
    )


# ============================================================
# 3. Y-axis
# ============================================================

ax.set_yticks(fold_dates["fold"])

ax.set_yticklabels(
    [f"Giai đoạn {i}" for i in fold_dates["fold"]],
    fontsize=10
)

# Giai đoạn 1 at the top
ax.invert_yaxis()

ax.set_ylabel("")


# ============================================================
# 4. X-axis
# ============================================================

ax.xaxis.set_major_locator(
    mdates.MonthLocator(bymonth=3, interval=1)
)

ax.xaxis.set_major_formatter(
    mdates.DateFormatter("%Y-%m")
)

ax.tick_params(
    axis="x",
    labelsize=9
)

ax.set_xlabel("Thời gian", fontsize=10)


# ============================================================
# 5. Grid
# ============================================================

ax.grid(
    axis="x",
    linestyle=":",
    linewidth=0.6,
    alpha=0.5
)

ax.grid(
    axis="y",
    visible=False
)


# ============================================================
# 6. Remove unnecessary borders
# ============================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


# ============================================================
# 7. Legend
# ============================================================

handles, labels = ax.get_legend_handles_labels()

ax.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.08),
    ncol=3,
    frameon=False,
    fontsize=9
)


# ============================================================
# 8. Layout
# ============================================================

plt.tight_layout()

plt.show()

In [ ]:
plt.savefig(
    "/kaggle/working/walk_forward_demonstration.png",
    dpi=400,
    bbox_inches="tight"
)